<a href="https://colab.research.google.com/github/NishanRegmi/machine-learning/blob/main/optuna_hyperparameter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [21]:
import optuna
import pandas as pd
import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [22]:
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
columns = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI',
           'DiabetesPedigreeFunction', 'Age', 'Outcome']

In [23]:
df = pd.read_csv(url, names=columns)

In [24]:
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [25]:
df.isnull().sum()

,0
Pregnancies,0
Glucose,0
BloodPressure,0
SkinThickness,0
Insulin,0
BMI,0
DiabetesPedigreeFunction,0
Age,0
Outcome,0


Insulin have 0 as missing value so first fill insulin with mean

In [28]:
# Replace zero values with NaN in columns where zero is not a valid value
cols_with_missing_vals = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
df[cols_with_missing_vals] = df[cols_with_missing_vals].replace(0, np.nan)

df.fillna(df.mean(), inplace=True)

In [29]:
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148.0,72.0,35.00000,155.548223,33.6,0.627,50,1
1,1,85.0,66.0,29.00000,155.548223,26.6,0.351,31,0
2,8,183.0,64.0,29.15342,155.548223,23.3,0.672,32,1
3,1,89.0,66.0,23.00000,94.000000,28.1,0.167,21,0
4,0,137.0,40.0,35.00000,168.000000,43.1,2.288,33,1


In [39]:
X = df.drop(columns=['Outcome'], axis=1)
y = df['Outcome']

In [40]:
#split into train test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=41)

In [41]:
#normalize the data using standardscaler
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [42]:
X_train

array([[ 0.69026007,  1.46386914,  0.11413899, ..., -0.82009461,
        -0.5036029 ,  2.96450972],
       [ 0.07144595,  0.09075822, -0.22019901, ..., -0.49400992,
         1.95122858,  1.08595408],
       [-0.85677523,  2.36811291, -1.89188903, ..., -0.91933777,
         0.52216597, -0.79260155],
       ...,
       [ 0.69026007, -0.11018484, -1.89188903, ..., -0.74920663,
         2.45972939,  0.01249372],
       [-0.23796111, -0.34461841,  0.11413899, ..., -0.11121486,
        -0.81630167, -0.70314652],
       [-0.23796111, -0.3111279 , -2.39339604, ..., -1.41555359,
        -0.98287952, -0.97151161]])

In [43]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

In [44]:
#defining the objective function
def objective(trial):
  #possible value for hyperparameter
  n_estimators = trial.suggest_int('n_estimators', 50, 200)
  max_depth = trial.suggest_int('max_depth', 3, 20)

  #create a model with possible hyperparameter
  model = RandomForestClassifier(
      n_estimators=n_estimators,
      max_depth=max_depth,
      random_state=41
  )
  score = np.mean(cross_val_score(model, X_train, y_train, cv=10, scoring='accuracy'))
  return score

In [46]:
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler())
study.optimize(objective, n_trials=50)

[I 2026-05-18 02:19:45,029] A new study created in memory with name: no-name-465cde4a-faeb-4a98-bd35-36dd92800295
[I 2026-05-18 02:19:47,387] Trial 0 finished with value: 0.7560796645702306 and parameters: {'n_estimators': 118, 'max_depth': 19}. Best is trial 0 with value: 0.7560796645702306.
[I 2026-05-18 02:19:48,905] Trial 1 finished with value: 0.7524109014675053 and parameters: {'n_estimators': 77, 'max_depth': 15}. Best is trial 0 with value: 0.7560796645702306.
[I 2026-05-18 02:19:52,554] Trial 2 finished with value: 0.763591893780573 and parameters: {'n_estimators': 196, 'max_depth': 8}. Best is trial 2 with value: 0.763591893780573.
[I 2026-05-18 02:19:54,882] Trial 3 finished with value: 0.7673305380852551 and parameters: {'n_estimators': 139, 'max_depth': 4}. Best is trial 3 with value: 0.7673305380852551.
[I 2026-05-18 02:19:56,505] Trial 4 finished with value: 0.7655136268343815 and parameters: {'n_estimators': 85, 'max_depth': 8}. Best is trial 3 with value: 0.76733053808

In [47]:
print("Best accuracy: ", study.best_trial.value)
print("Best parameter: ", study.best_trial.params)

Best accuracy:  0.7729559748427673
Best parameter:  {'n_estimators': 122, 'max_depth': 4}


In [48]:
#using the best parameter value, again train the model

classifier = RandomForestClassifier(n_estimators=122, max_depth=4, random_state=41)

In [49]:
classifier.fit(X_train, y_train)
y_pred = classifier.predict(X_test)

In [50]:
from sklearn.metrics import accuracy_score
print("Accuracy: ", accuracy_score(y_test, y_pred))

Accuracy:  0.7402597402597403


**Samplers in Optuna**

In [51]:
#defining the objective function
def objective(trial):
  #possible value for hyperparameter
  n_estimators = trial.suggest_int('n_estimators', 50, 200)
  max_depth = trial.suggest_int('max_depth', 3, 20)

  #create a model with possible hyperparameter
  model = RandomForestClassifier(
      n_estimators=n_estimators,
      max_depth=max_depth,
      random_state=41
  )
  score = np.mean(cross_val_score(model, X_train, y_train, cv=10, scoring='accuracy'))
  return score

In [52]:
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.RandomSampler())
study.optimize(objective, n_trials=50)

[I 2026-05-18 02:34:11,598] A new study created in memory with name: no-name-0fd8e537-ecd2-4db8-9738-d6fed842fde7
[I 2026-05-18 02:34:15,408] Trial 0 finished with value: 0.7598183088749126 and parameters: {'n_estimators': 196, 'max_depth': 10}. Best is trial 0 with value: 0.7598183088749126.
[I 2026-05-18 02:34:16,477] Trial 1 finished with value: 0.7635918937805731 and parameters: {'n_estimators': 53, 'max_depth': 11}. Best is trial 1 with value: 0.7635918937805731.
[I 2026-05-18 02:34:19,081] Trial 2 finished with value: 0.7673305380852551 and parameters: {'n_estimators': 141, 'max_depth': 7}. Best is trial 2 with value: 0.7673305380852551.
[I 2026-05-18 02:34:20,520] Trial 3 finished with value: 0.7579664570230608 and parameters: {'n_estimators': 72, 'max_depth': 14}. Best is trial 2 with value: 0.7673305380852551.
[I 2026-05-18 02:34:22,764] Trial 4 finished with value: 0.7523759608665269 and parameters: {'n_estimators': 113, 'max_depth': 12}. Best is trial 2 with value: 0.7673305

In [53]:
print(f'Best accuracy: {study.best_trial.value}')
print(f'Best parameter: {study.best_trial.params}')

Best accuracy: 0.7766247379454926
Best parameter: {'n_estimators': 193, 'max_depth': 4}


**Optuna Visualizations**

In [54]:
from optuna.visualization import plot_optimization_history, plot_parallel_coordinate, plot_slice, plot_contour, plot_param_importances

In [55]:
#optimization history
plot_optimization_history(study).show()

In [56]:
#parallel cordinates plot
plot_parallel_coordinate(study).show()

In [57]:
#slice plot
plot_slice(study).show()

In [58]:
#contour plot
plot_contour(study).show()

In [59]:
#important parameter
plot_param_importances(study).show()

**Optimiziing multiple ML model**

In [60]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

In [62]:
def objective(trial):
  #define multiple model
  classifier_name = trial.suggest_categorical('classifier', ['svm', 'randomforest', 'gradientboost'])

  if classifier_name == 'svm':
    c = trial.suggest_float('C', 0.1, 100, log=True)
    kernel = trial.suggest_categorical('kernel', ['linear', 'rbf', 'poly', 'sigmoid'])
    gamma = trial.suggest_categorical('gamma', ['scale', 'auto'])

    model = SVC(C=c, kernel=kernel, gamma=gamma, random_state=42)

  elif classifier_name == 'randomforest':
    n_estimators = trial.suggest_int('n_estimators', 50, 300)
    max_depth = trial.suggest_int('max_depth', 3, 20)
    min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
    min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)
    bootstrap = trial.suggest_categorical('bootstrap', [True, False])

    model = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            bootstrap=bootstrap,
            random_state=42)
  elif classifier_name == 'gradientboost':
    n_estimators = trial.suggest_int('n_estimators', 50, 300)
    learning_rate = trial.suggest_float('learning_rate', 0.01, 0.3, log=True)
    max_depth = trial.suggest_int('max_depth', 3, 20)
    min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
    min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)

    model = GradientBoostingClassifier(
            n_estimators=n_estimators,
            learning_rate=learning_rate,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            random_state=42)

  score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()
  return score

In [63]:
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50)

[I 2026-05-18 03:04:19,263] A new study created in memory with name: no-name-e31b7057-f43d-47d1-9d18-e3f65152230f
[I 2026-05-18 03:04:20,451] Trial 0 finished with value: 0.7541899441340782 and parameters: {'classifier': 'gradientboost', 'n_estimators': 116, 'learning_rate': 0.12248312943545496, 'max_depth': 9, 'min_samples_split': 5, 'min_samples_leaf': 8}. Best is trial 0 with value: 0.7541899441340782.
[I 2026-05-18 03:04:20,474] Trial 1 finished with value: 0.7765363128491619 and parameters: {'classifier': 'svm', 'C': 0.4049644250283043, 'kernel': 'linear', 'gamma': 'auto'}. Best is trial 1 with value: 0.7765363128491619.
[I 2026-05-18 03:04:20,500] Trial 2 finished with value: 0.7392923649906891 and parameters: {'classifier': 'svm', 'C': 3.168322941081726, 'kernel': 'poly', 'gamma': 'auto'}. Best is trial 1 with value: 0.7765363128491619.
[I 2026-05-18 03:04:22,303] Trial 3 finished with value: 0.7560521415270017 and parameters: {'classifier': 'gradientboost', 'n_estimators': 161,

In [64]:
print('Best accuracy', study.best_trial.value)
print("Best param", study.best_trial.params)

Best accuracy 0.7802607076350093
Best param {'classifier': 'svm', 'C': 0.1074710140811938, 'kernel': 'linear', 'gamma': 'auto'}


In [65]:
study.trials_dataframe()

,number,value,datetime_start,datetime_complete,duration,params_C,params_bootstrap,params_classifier,params_gamma,params_kernel,params_learning_rate,params_max_depth,params_min_samples_leaf,params_min_samples_split,params_n_estimators,state
0,0,0.754190,2026-05-18 03:04:19.265019,2026-05-18 03:04:20.451729,0 days 00:00:01.186710,NaN,NaN,gradientboost,NaN,NaN,0.122483,9.0,8.0,5.0,116.0,COMPLETE
1,1,0.776536,2026-05-18 03:04:20.452674,2026-05-18 03:04:20.474117,0 days 00:00:00.021443,0.404964,NaN,svm,auto,linear,NaN,NaN,NaN,NaN,NaN,COMPLETE
2,2,0.739292,2026-05-18 03:04:20.475038,2026-05-18 03:04:20.500860,0 days 00:00:00.025822,3.168323,NaN,svm,auto,poly,NaN,NaN,NaN,NaN,NaN,COMPLETE
3,3,0.756052,2026-05-18 03:04:20.501881,2026-05-18 03:04:22.303229,0 days 00:00:01.801348,NaN,NaN,gradientboost,NaN,NaN,0.126203,15.0,9.0,7.0,161.0,COMPLETE
4,4,0.759777,2026-05-18 03:04:22.304185,2026-05-18 03:04:23.581000,0 days 00:00:01.276815,NaN,True,randomforest,NaN,NaN,NaN,4.0,6.0,4.0,269.0,COMPLETE
5,5,0.703911,2026-05-18 03:04:23.581979,2026-05-18 03:04:23.605922,0 days 00:00:00.023943,14.927726,NaN,svm,scale,sigmoid,NaN,NaN,NaN,NaN,NaN,COMPLETE
6,6,0.759777,2026-05-18 03:04:23.606788,2026-05-18 03:04:24.763842,0 days 00:00:01.157054,NaN,True,randomforest,NaN,NaN,NaN,4.0,6.0,9.0,239.0,COMPLETE
7,7,0.756052,2026-05-18 03:04:24.764803,2026-05-18 03:04:25.263839,0 days 00:00:00.499036,NaN,False,randomforest,NaN,NaN,NaN,4.0,2.0,9.0,121.0,COMPLETE
8,8,0.696462,2026-05-18 03:04:25.264985,2026-05-18 03:04:25.287197,0 days 00:00:00.022212,0.121980,NaN,svm,auto,poly,NaN,NaN,NaN,NaN,NaN,COMPLETE
9,9,0.748603,2026-05-18 03:04:25.288229,2026-05-18 03:04:25.634196,0 days 00:00:00.345967,NaN,False,randomforest,NaN,NaN,NaN,5.0,6.0,5.0,81.0,COMPLETE


In [66]:
study.trials_dataframe()['params_classifier'].value_counts()

,count
params_classifier,
svm,36
gradientboost,7
randomforest,7


In [67]:
study.trials_dataframe().groupby('params_classifier')['value'].mean()

,value
params_classifier,
gradientboost,0.749667
randomforest,0.758446
svm,0.764484


In [68]:
#plot history
plot_optimization_history(study).show()

In [69]:
plot_slice(study).show()

In [70]:
plot_param_importances(study).show()